# Parallax 3D GIF Generator

Turn any single image into a 3D parallax GIF using monocular depth estimation, subject segmentation, and LaMa inpainting.

**Pipeline:**
1. Depth estimation (Depth Anything V2)
2. Subject segmentation (RMBG-2.0)
3. Background inpainting (LaMa)
4. Orbital parallax rendering
5. GIF assembly

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/YOUR_USERNAME/parallax-3d-gif/blob/main/parallax_3d_gif.ipynb)

## 1. Setup

Install dependencies and upload your image.

In [ ]:
!pip install -q torch torchvision opencv-python-headless Pillow scipy tqdm huggingface_hub "transformers>=4.45,<5.0"
!pip install -q simple-lama-inpainting kornia

In [ ]:
from huggingface_hub import login
login()

In [ ]:
from google.colab import files

uploaded = files.upload()
INPUT = list(uploaded.keys())[0]
print(f'\nReady: {INPUT}')

## 2. Configuration

Adjust these parameters to control the 3D effect.

In [ ]:
from pathlib import Path

# -- Animation --
FRAMES        = 9        # number of viewpoints to render
ARC_DEG       = 7.0      # total camera arc in degrees
FPS           = 12       # GIF frame rate
PING_PONG     = True     # bounce animation back and forth

# -- Image --
MAX_DIM       = 2048     # max image dimension (longer side)

# -- Parallax --
BG_PARALLAX   = 2.5      # background shift multiplier vs foreground
BG_BLUR       = 3        # simulated depth-of-field blur on background

# -- Compositing --
AO_WIDTH      = 6        # ambient occlusion shadow width
AO_STRENGTH   = 0.35     # ambient occlusion shadow intensity

# -- Inpainting --
MASK_DILATE_K = 21       # dilation kernel size for inpaint mask
MASK_DILATE_I = 5        # dilation iterations

OUTPUT = Path(INPUT).stem + '_3d.gif'

## 3. Run Pipeline

In [ ]:
import cv2
import torch
import numpy as np
import time
import os
import warnings
warnings.filterwarnings('ignore')

from PIL import Image
from tqdm import tqdm
from IPython.display import display, Image as IPImage

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Device: {device}')

In [ ]:
# ── Helper functions ──

def guided_filter(guide, target, radius=8, eps=0.01):
    """Edge-preserving smoothing using a guided filter."""
    guide = guide.astype(np.float64)
    target = target.astype(np.float64)
    ksize = 2 * radius + 1
    mean_g = cv2.boxFilter(guide, -1, (ksize, ksize))
    mean_t = cv2.boxFilter(target, -1, (ksize, ksize))
    mean_gt = cv2.boxFilter(guide * target, -1, (ksize, ksize))
    mean_gg = cv2.boxFilter(guide * guide, -1, (ksize, ksize))
    a = (mean_gt - mean_g * mean_t) / (mean_gg - mean_g * mean_g + eps)
    b = mean_t - a * mean_g
    mean_a = cv2.boxFilter(a, -1, (ksize, ksize))
    mean_b = cv2.boxFilter(b, -1, (ksize, ksize))
    return (mean_a * guide + mean_b).astype(np.float32)


def iterative_inpaint(img_bgr, m, passes=5):
    """Multi-pass OpenCV inpainting for large masked areas."""
    result = img_bgr.copy()
    remaining = m.copy()
    k = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (7, 7))
    for _ in range(passes):
        eroded = cv2.erode(remaining, k, iterations=3)
        band = np.clip(remaining.astype(np.float32) - eroded.astype(np.float32), 0, 1).astype(np.uint8)
        if band.sum() < 10:
            break
        result = cv2.inpaint(result, band, 7, cv2.INPAINT_NS)
        remaining = eroded
    if remaining.sum() > 0:
        result = cv2.inpaint(result, remaining, 15, cv2.INPAINT_TELEA)
    return result


def inpaint_lama(img_rgb, mask_binary):
    """LaMa inpainting with OpenCV fallback."""
    try:
        from simple_lama_inpainting import SimpleLama
        print('  Using LaMa inpainting...')
        lama = SimpleLama()
        img_pil = Image.fromarray(img_rgb)
        mask_pil = Image.fromarray((mask_binary * 255).astype(np.uint8))
        result_pil = lama(img_pil, mask_pil)
        return np.array(result_pil)
    except Exception as e:
        print(f'  LaMa failed ({e}), falling back to OpenCV...')
        img_bgr = cv2.cvtColor(img_rgb, cv2.COLOR_RGB2BGR)
        result_bgr = iterative_inpaint(img_bgr, mask_binary.astype(np.uint8))
        return cv2.cvtColor(result_bgr, cv2.COLOR_BGR2RGB)


def build_ao_shadow(mask, width=6, strength=0.35):
    """Generate ambient occlusion shadow at mask edges."""
    dist = cv2.distanceTransform((mask > 0.5).astype(np.uint8), cv2.DIST_L2, 5)
    dist = np.clip(dist / max(width, 1), 0, 1)
    ao = 1.0 - (1.0 - dist) * strength * (mask > 0.5).astype(np.float32)
    return cv2.GaussianBlur(np.clip(ao, 0, 1).astype(np.float32), (5, 5), 1.5)

In [ ]:
# ── Load image ──

raw = Image.open(INPUT).convert('RGB')
w, h = raw.size
if max(w, h) > MAX_DIM:
    s = MAX_DIM / max(w, h)
    w, h = int(w * s) & ~1, int(h * s) & ~1
    raw = raw.resize((w, h), Image.LANCZOS)
img = np.array(raw)
H, W = img.shape[:2]
print(f'Image: {W}x{H}')

In [ ]:
# ── Stage 1: Depth Estimation ──

t0 = time.time()
print('Stage 1: Depth Estimation')

from transformers import AutoImageProcessor, AutoModelForDepthEstimation

depth_proc = AutoImageProcessor.from_pretrained('depth-anything/Depth-Anything-V2-Small-hf')
depth_model = AutoModelForDepthEstimation.from_pretrained(
    'depth-anything/Depth-Anything-V2-Small-hf'
).to(device).eval()

inputs = depth_proc(images=Image.fromarray(img), return_tensors='pt').to(device)
with torch.no_grad():
    pred = depth_model(**inputs).predicted_depth

depth_raw = torch.nn.functional.interpolate(
    pred.unsqueeze(1), size=(H, W), mode='bicubic'
)[0, 0].cpu().numpy().astype(np.float32)
depth = (depth_raw - depth_raw.min()) / (depth_raw.max() - depth_raw.min() + 1e-8)

del depth_model, depth_proc
torch.cuda.empty_cache()

# Auto-detect polarity (near=0, far=1)
center_d = depth[H//3:2*H//3, W//3:2*W//3].mean()
edge_d = np.concatenate([depth[0,:], depth[-1,:], depth[:,0], depth[:,-1]]).mean()
if center_d > edge_d:
    depth = 1.0 - depth
    print('  (inverted depth polarity)')

# Refine depth edges
gray = cv2.cvtColor(img, cv2.COLOR_RGB2GRAY).astype(np.float32) / 255.0
depth = guided_filter(gray, depth, radius=8, eps=0.02)
depth = (depth - depth.min()) / (depth.max() - depth.min() + 1e-8)

print(f'  Done in {time.time()-t0:.1f}s')
cv2.imwrite('depth.png', cv2.applyColorMap((depth * 255).astype(np.uint8), cv2.COLORMAP_INFERNO))

In [ ]:
# ── Stage 2: Subject Segmentation ──

t0 = time.time()
print('Stage 2: Segmentation')

from transformers import AutoModelForImageSegmentation
from torchvision.transforms.functional import normalize

rmbg = AutoModelForImageSegmentation.from_pretrained(
    'briaai/RMBG-2.0', trust_remote_code=True
).to(device).eval()

t_img = torch.from_numpy(img).permute(2, 0, 1).float() / 255.0
t_img = normalize(t_img, [0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
t_img = torch.nn.functional.interpolate(
    t_img.unsqueeze(0), (1024, 1024), mode='bilinear'
).to(device)

with torch.no_grad():
    mask = torch.sigmoid(rmbg(t_img)[-1][0, 0]).cpu().numpy()
mask = cv2.resize(mask, (W, H))
mask = (mask > 0.5).astype(np.float32)

kern = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (5, 5))
mask = cv2.morphologyEx(mask, cv2.MORPH_CLOSE, kern, iterations=3)
mask = cv2.morphologyEx(mask, cv2.MORPH_OPEN, kern, iterations=1)

del rmbg
torch.cuda.empty_cache()

print(f'  Foreground: {mask.mean()*100:.1f}%  Done in {time.time()-t0:.1f}s')
cv2.imwrite('mask.png', (mask * 255).astype(np.uint8))

In [ ]:
# ── Stage 3: Background Inpainting ──

t0 = time.time()
print('Stage 3: Background Inpainting')

# Aggressively dilate the mask to remove ghost edges
inpaint_mask = cv2.dilate(
    mask.astype(np.uint8),
    cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (MASK_DILATE_K, MASK_DILATE_K)),
    iterations=MASK_DILATE_I
)
print(f'  Inpaint mask covers {inpaint_mask.mean()*100:.1f}% of image')

# Inpaint color
bg_image = inpaint_lama(img, inpaint_mask)
if bg_image.shape[:2] != (H, W):
    bg_image = cv2.resize(bg_image, (W, H), interpolation=cv2.INTER_LANCZOS4)

# Simulated depth-of-field
if BG_BLUR > 0:
    blur_k = BG_BLUR * 2 + 1
    bg_image = cv2.GaussianBlur(bg_image, (blur_k, blur_k), BG_BLUR * 0.8)

# Inpaint depth
d_u8 = (depth * 255).astype(np.uint8)
d_u8[inpaint_mask > 0] = 0
bg_depth = cv2.inpaint(d_u8, inpaint_mask, 15, cv2.INPAINT_NS).astype(np.float32) / 255.0

# Ambient occlusion
ao_map = build_ao_shadow(mask, width=AO_WIDTH, strength=AO_STRENGTH)

print(f'  Done in {time.time()-t0:.1f}s')
Image.fromarray(bg_image).save('bg_inpainted.png')

In [ ]:
# ── Stage 4: Orbital Rendering ──

t0 = time.time()
print('Stage 4: Orbital Rendering')

max_shift = W * np.tan(np.radians(ARC_DEG)) * 0.3
disparity_fg = (1.0 - depth) * max_shift
disparity_bg = (1.0 - bg_depth) * max_shift * BG_PARALLAX
u_coords = np.arange(W, dtype=np.float32)
angles = np.linspace(-1.0, 1.0, FRAMES)

# Subject centroid anchor
fg_ys, fg_xs = np.where(mask > 0.5)
if len(fg_xs) > 0:
    subject_center_disparity = (1.0 - np.median(depth[fg_ys, fg_xs])) * max_shift
else:
    subject_center_disparity = 0.0

frames = []
for t_val in tqdm(angles, desc='Rendering'):
    anchor_offset = subject_center_disparity * t_val

    # Warp background
    result = bg_image.copy().astype(np.float32)
    bg_shift = disparity_bg * t_val - anchor_offset
    for row in range(H):
        su = np.clip(u_coords + bg_shift[row], 0, W - 1).astype(int)
        result[row] = bg_image[row, su]

    # Forward-warp foreground with z-buffering
    fg_frame = np.zeros((H, W, 3), dtype=np.float32)
    fg_alpha = np.zeros((H, W), dtype=np.float32)
    fg_zbuf = np.full((H, W), np.inf, dtype=np.float32)
    shift = disparity_fg * t_val - anchor_offset

    for row in range(H):
        dst_u = u_coords + shift[row]
        for col in range(W):
            if mask[row, col] < 0.5:
                continue
            du = int(round(dst_u[col]))
            if 0 <= du < W and depth[row, col] < fg_zbuf[row, du]:
                fg_zbuf[row, du] = depth[row, col]
                fg_frame[row, du] = img[row, col].astype(np.float32)
                fg_alpha[row, du] = mask[row, col]

    # Fill cracks
    cracks = cv2.dilate((fg_alpha > 0).astype(np.uint8), np.ones((3, 1), np.uint8))
    crack_mask = np.clip(cracks - (fg_alpha > 0).astype(np.uint8), 0, 1).astype(np.uint8)
    if crack_mask.sum() > 0:
        filled = cv2.inpaint(fg_frame.astype(np.uint8), crack_mask, 3, cv2.INPAINT_TELEA)
        fg_frame = np.where(crack_mask[:, :, None] > 0, filled.astype(np.float32), fg_frame)
        fg_alpha = np.maximum(fg_alpha, crack_mask.astype(np.float32))

    # Apply ambient occlusion
    for c in range(3):
        fg_frame[:, :, c] *= np.where(fg_alpha > 0.5, ao_map, 1.0)

    # Composite with eroded alpha
    erode_kern = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (3, 3))
    alpha = cv2.GaussianBlur(
        cv2.erode(fg_alpha, erode_kern, iterations=1), (9, 9), 2.5
    )[:, :, None]
    result = result * (1.0 - alpha) + fg_frame * alpha
    frames.append(np.clip(result, 0, 255).astype(np.uint8))

print(f'  {len(frames)} frames  Done in {time.time()-t0:.1f}s')

In [ ]:
# ── Stage 5: GIF Assembly ──

print('Stage 5: GIF Assembly')

if PING_PONG and len(frames) > 2:
    gif_frames = frames + frames[-2:0:-1]
else:
    gif_frames = frames

pil_frames = [Image.fromarray(f) for f in gif_frames]
pil_frames[0].save(
    OUTPUT, save_all=True, append_images=pil_frames[1:],
    duration=int(1000 / FPS), loop=0, optimize=True
)

size_mb = os.path.getsize(OUTPUT) / 1024**2
print(f'{OUTPUT}: {len(gif_frames)} frames, {size_mb:.1f} MB')

# Save individual frames
os.makedirs('frames', exist_ok=True)
for i, f in enumerate(frames):
    Image.fromarray(f).save(f'frames/frame_{i:03d}.png')
print(f'Saved {len(frames)} PNGs to frames/')

display(IPImage(filename=OUTPUT))

## 4. Results & Download

In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 4, figsize=(20, 5))
for ax, (title, path) in zip(axes, [
    ('Input', INPUT), ('Depth', 'depth.png'),
    ('Mask', 'mask.png'), ('BG Inpainted', 'bg_inpainted.png')
]):
    ax.imshow(Image.open(path))
    ax.set_title(title, fontsize=14)
    ax.axis('off')
plt.tight_layout()
plt.show()

In [ ]:
from google.colab import files
files.download(OUTPUT)

# Optional: download frames zip for lenticular interlacing
# import shutil
# shutil.make_archive('lenticular_frames', 'zip', 'frames')
# files.download('lenticular_frames.zip')